# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quangptt0910/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 2 — Refresh / Content Opportunity Scoring.** Skills: `hunting-leakage-and-validating`, `flyrank/flyrank-data`.

**Data used:** `data/raw/content_refresh_anonymized.csv` — the 30,000-row anonymized starter slice (32 pseudonymized clients). This is the same data Weeks 5–7 and the capstone paper use.

**Deviation from my ML-04 contract (stated honestly).** My Week 3 data contract planned a *warehouse* feature vector: March 2026 features → April 2026 label, strictly forward in time. The modeling weeks were built on the starter slice instead, so this notebook checks the features those weeks actually used. The warehouse contract is kept for a later forward-in-time version (Phase B).

**Label:** `is_declining_label = 1` when `trend_direction == 'down'`, i.e. impressions in the last 30 days fell more than 20% versus the 30 days before (days 31–60 back).

**Timeline of the starter slice** (days counted back from export):

```
day 90 ............ day 61 | day 60 ...... day 31 | day 30 ....... day 1
   strictly before label   |  label: "previous"   |  label: "last 30"
<------------------------- *_90d aggregates cover ALL of this ------------------------>
```

Every `*_90d` feature therefore *contains* the label's two windows. Section 3 measures how much that matters.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
N_SPLITS = 10          # client-grouped splits per leakage test
np.random.seed(RANDOM_SEED)

def find(rel):
    for base in [Path("."), Path(".."), Path("../..")]:
        if (base / rel).exists():
            return base / rel
    raise FileNotFoundError(rel)

raw = pd.read_csv(find("data/raw/content_refresh_anonymized.csv"))
print(f"Loaded starter slice: {len(raw):,} rows x {raw.shape[1]} columns, {raw['client_id'].nunique()} clients")

# Grain check: one row per (client, content)
assert not raw.duplicated(["client_id", "content_id"]).any()
print("Grain check passed: one row per (client_id, content_id)")

df = raw.copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Label base rate: {df['is_declining_label'].mean():.3f} ({df['is_declining_label'].sum():,} declining)")

# --- Missingness BEFORE filling, by content type (a blind fillna(0) can smuggle in a category signal)
fill_cols = ["word_count", "search_volume", "cpc", "competition", "scroll_rate"]
miss = df.groupby("content_type")[fill_cols].apply(lambda g: g.isna().mean()).round(3)
miss["rows"] = df["content_type"].value_counts()
print("\nShare missing by content_type (before any fill):")
print(miss.to_string())
print(f"\nPages with avg_position == 0 ('no position data', not rank zero): {(df['avg_position'] == 0).sum():,}")

# --- Engineered features (identical to w05 / w07 / capstone)
for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
df["has_word_count"] = (df["word_count"].fillna(0) > 0).astype(int)  # flag BEFORE filling, so missingness stays explicit
for col in fill_cols:
    df[col] = df[col].fillna(0)

NUMERIC_FEATURES = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "ctr",
    "days_with_impressions", "days_with_sessions", "avg_position",
    "has_position_data", "search_volume", "cpc", "competition",
    "content_age_days", "days_since_last_update", "word_count",
    "has_word_count", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = ["content_type", "main_intent", "competition_level",
                        "freshness_tier", "position_tier", "word_count_tier"]
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# --- Strict variant: only traffic from days 61-90, which lies entirely BEFORE the label window
for m in ["impressions", "clicks", "sessions"]:
    early = df[f"{m}_90d"].fillna(0) - df[f"{m}_last_30d"].fillna(0) - df[f"{m}_prev_30d"].fillna(0)
    assert (early >= 0).all(), f"{m}: 90d total smaller than last+prev 30d"
    df[f"log_{m}_d61_90"] = np.log1p(early)
STATIC_NUMERIC = ["search_volume", "cpc", "competition", "content_age_days",
                  "days_since_last_update", "word_count", "has_word_count"]
STATIC_CATEGORICAL = ["content_type", "main_intent", "competition_level", "freshness_tier", "word_count_tier"]
STRICT_NUMERIC = STATIC_NUMERIC + ["log_impressions_d61_90", "log_clicks_d61_90", "log_sessions_d61_90"]

X = df[FEATURES]
assert not X[NUMERIC_FEATURES].isna().any().any(), "NaN left in numeric features"
print(f"\nFeature vector: {X.shape[0]:,} rows x {len(FEATURES)} features "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical); no NaNs in numeric columns")
print(f"Strict variant: {len(STRICT_NUMERIC)} numeric + {len(STATIC_CATEGORICAL)} categorical, all from before the label window or static")


Loaded starter slice: 30,000 rows x 44 columns, 32 clients
Grain check passed: one row per (client_id, content_id)
Label base rate: 0.542 (16,262 declining)

Share missing by content_type (before any fill):
                    word_count  search_volume    cpc  competition  scroll_rate   rows
content_type                                                                         
comparison article       0.000          0.000  0.000        0.000        0.003    697
feedly article           0.000          1.000  1.000        1.000        0.000   2096
keyword article          0.283          0.014  0.014        0.014        0.004  27207

Pages with avg_position == 0 ('no position data', not rank zero): 1,205

Feature vector: 30,000 rows x 24 features (18 numeric, 6 categorical); no NaNs in numeric columns
Strict variant: 10 numeric + 5 categorical, all from before the label window or static


**What I see.** The fills are not neutral: `word_count`, `search_volume`, `cpc` and `competition` are missing at very different rates across content types, so a plain `fillna(0)` would partly encode content type. I keep an explicit `has_word_count` flag (computed before filling) and also give the model `content_type` directly, so the signal is visible rather than hidden in zeros. `avg_position == 0` means "no data", so it gets its own `has_position_data` flag.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
MEANING = {
    "log_impressions_90d": "log(1+GSC impressions, 90d)", "log_clicks_90d": "log(1+GSC clicks, 90d)",
    "log_sessions_90d": "log(1+GA4 sessions, 90d)", "ctr": "clicks/impressions x100 (0.76 = 0.76%)",
    "days_with_impressions": "days in 90d with >=1 impression", "days_with_sessions": "days in 90d with >=1 session",
    "avg_position": "mean GSC position over 90d (0 = no data)", "has_position_data": "1 if avg_position > 0",
    "search_volume": "keyword monthly search volume", "cpc": "keyword cost per click",
    "competition": "keyword competition index", "content_age_days": "days since page was created (at export)",
    "days_since_last_update": "days since last edit (at export)", "word_count": "words on the page (at export)",
    "has_word_count": "1 if word_count was recorded", "engagement_rate": "engaged/total sessions x100, 90d",
    "scroll_rate": "scroll events/pageviews x100, 90d (can exceed 100)", "ai_traffic_pct": "AI-referred share of sessions x100, 90d",
    "content_type": "page type bucket", "main_intent": "search intent bucket",
    "competition_level": "keyword competition bucket", "freshness_tier": "bucket of days_since_last_update",
    "position_tier": "bucket of avg_position", "word_count_tier": "bucket of word_count",
}
WINDOW_90D = {"log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "ctr", "days_with_impressions",
              "days_with_sessions", "avg_position", "has_position_data", "engagement_rate", "scroll_rate",
              "ai_traffic_pct", "position_tier"}
AT_EXPORT = {"content_age_days", "days_since_last_update", "freshness_tier", "word_count", "has_word_count", "word_count_tier"}

def timing(f):
    if f in WINDOW_90D:
        return "OVERLAPS label window (90d includes last 30d)"
    if f in AT_EXPORT:
        return "measured at export; an edit inside the label window would show here"
    return "static keyword/page metadata"

def handling(f):
    if f in fill_cols: return "NaN -> 0, paired with has_ flag / content_type"
    if f.startswith("log_"): return "NaN -> 0 then log1p"
    if f in CATEGORICAL_FEATURES: return "NaN -> 'unknown', one-hot"
    if f == "avg_position": return "0 kept; has_position_data flags it"
    return "none needed"

notes = pd.DataFrame([{
    "feature": f,
    "type": "categorical" if f in CATEGORICAL_FEATURES else "numeric",
    "meaning": MEANING[f],
    "missing_raw_%": round(100 * raw[f].isna().mean(), 1) if f in raw.columns else round(100 * raw[f.replace("log_", "")].isna().mean(), 1)
                      if f.replace("log_", "") in raw.columns else 0.0,
    "missing_handling": handling(f),
    "available_when": timing(f),
} for f in FEATURES])
pd.set_option("display.max_colwidth", 70)
print(notes.to_string(index=False))
print("\nTiming summary:", notes["available_when"].str.split(" ").str[0].value_counts().to_dict())


               feature        type                                            meaning  missing_raw_%                               missing_handling                                                      available_when
   log_impressions_90d     numeric                        log(1+GSC impressions, 90d)            0.0                            NaN -> 0 then log1p                       OVERLAPS label window (90d includes last 30d)
        log_clicks_90d     numeric                             log(1+GSC clicks, 90d)            0.0                            NaN -> 0 then log1p                       OVERLAPS label window (90d includes last 30d)
      log_sessions_90d     numeric                           log(1+GA4 sessions, 90d)            0.0                            NaN -> 0 then log1p                       OVERLAPS label window (90d includes last 30d)
                   ctr     numeric             clicks/impressions x100 (0.76 = 0.76%)            0.0                                    

**What this table says.** Twelve of the 24 features are 90-day aggregates, which **overlap the label window**. That is not label leakage in the strict sense (none of them *is* the label), but they partly describe the same days the label describes. Six more are measured at export time, so an edit made inside the label window would already show up in `days_since_last_update`. Only the keyword metadata is truly static. Section 3 measures the effect instead of guessing.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Five attacks, each with code: **(a)** single-column scan for label-derived columns, **(b)** positive control, **(c)** overlapping windows, **(d)** product flags, **(e)** client memorization. Every model test uses the same model as the paper, scored on **client-grouped** splits (10 splits) with the base rate printed alongside.


In [3]:
# (a) Single-column scan: how well does each RAW numeric column alone rank the label?
num_raw = [c for c in raw.select_dtypes("number").columns]
scan = []
for c in num_raw:
    v = raw[c].fillna(raw[c].median())
    if v.nunique() < 2:
        continue
    auc = roc_auc_score(df["is_declining_label"], v)
    scan.append({"column": c, "single_column_AUC": round(max(auc, 1 - auc), 3),
                 "in_feature_set": c in FEATURES or f"log_{c}" in FEATURES})
scan = pd.DataFrame(scan).sort_values("single_column_AUC", ascending=False)
print(scan.head(12).to_string(index=False))
suspects = scan[scan["single_column_AUC"] >= 0.75]
print(f"\nColumns that alone reach AUC >= 0.75: {suspects['column'].tolist()}")
assert not suspects["in_feature_set"].any(), "A near-perfect single column is in the feature set!"
print("None of them is in the feature set.")


               column  single_column_AUC  in_feature_set
            trend_pct              0.955           False
 impressions_prev_30d              0.621           False
     content_age_days              0.591            True
       age_tier_order              0.585           False
      impressions_90d              0.584            True
days_with_impressions              0.579            True
        search_volume              0.560            True
    sessions_prev_30d              0.542           False
           word_count              0.534            True
    sessions_last_30d              0.533           False
         avg_position              0.530            True
           char_count              0.528           False

Columns that alone reach AUC >= 0.75: ['trend_pct']
None of them is in the feature set.


In [4]:
# Shared harness: same model as the paper, client-grouped splits
def precision_at_k(y, s, k):
    return float(np.asarray(y)[np.argsort(-np.asarray(s))[:k]].mean())

def make_model(num, cat):
    prep = ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
    return Pipeline([("prep", prep), ("clf", HistGradientBoostingClassifier(
        max_depth=5, min_samples_leaf=25, class_weight="balanced", random_state=RANDOM_SEED))])

splits = list(GroupShuffleSplit(n_splits=N_SPLITS, test_size=0.20, random_state=RANDOM_SEED)
              .split(df, groups=df["client_id"]))

def evaluate(name, num, cat):
    rows = []
    for tr, te in splits:
        a, b = df.iloc[tr], df.iloc[te]
        s = make_model(num, cat).fit(a[num + cat], a["is_declining_label"]).predict_proba(b[num + cat])[:, 1]
        y = b["is_declining_label"]
        rows.append({"AUC": roc_auc_score(y, s), "P@10": precision_at_k(y, s, 10),
                     "P@50": precision_at_k(y, s, 50), "base_rate": y.mean()})
    r = pd.DataFrame(rows)
    out = {"feature_set": name, **{f"{m}_mean": r[m].mean() for m in r}, "AUC_sd": r["AUC"].std()}
    print(f"{name:<44} AUC {out['AUC_mean']:.3f} ± {out['AUC_sd']:.3f} | P@10 {out['P@10_mean']:.2f} | "
          f"P@50 {out['P@50_mean']:.2f} | base rate {out['base_rate_mean']:.2f}")
    return out

for c in ["impressions_last_30d", "impressions_prev_30d"]:
    df[c + "_raw"] = df[c].fillna(0)

print(f"(b) Positive control + (c) overlapping windows — {N_SPLITS} client-grouped splits\n")
results = [
    evaluate("LEAK on purpose: + impressions last/prev 30d", NUMERIC_FEATURES + ["impressions_last_30d_raw", "impressions_prev_30d_raw"], CATEGORICAL_FEATURES),
    evaluate("Paper features (90d windows + static)", NUMERIC_FEATURES, CATEGORICAL_FEATURES),
    evaluate("Strict: days 61-90 traffic + static", STRICT_NUMERIC, STATIC_CATEGORICAL),
    evaluate("Static only (no traffic at all)", STATIC_NUMERIC, STATIC_CATEGORICAL),
]

# Week 4 rule on the same splits, for reference
rule = (((df["avg_position"] >= 4) & (df["avg_position"] <= 20)).astype(int)
        * (df["days_with_impressions"] >= 45).astype(int)
        * (df["impressions_90d"] >= 300).astype(int) * df["impressions_90d"])
rr = [{"AUC": roc_auc_score(df["is_declining_label"].iloc[te], rule.iloc[te]),
       "P@10": precision_at_k(df["is_declining_label"].iloc[te], rule.iloc[te], 10),
       "P@50": precision_at_k(df["is_declining_label"].iloc[te], rule.iloc[te], 50)} for _, te in splits]
rr = pd.DataFrame(rr).mean()
print(f"{'Week 4 rule (reference)':<44} AUC {rr['AUC']:.3f}         | P@10 {rr['P@10']:.2f} | P@50 {rr['P@50']:.2f}")

leak, paper, strict, static = results
assert leak["AUC_mean"] > 0.95, "Positive control failed to light up: the harness is not sensitive"
assert paper["AUC_mean"] < 0.85, "Paper features look suspiciously strong"
print("\nPositive control lit up (harness works); the clean feature set stays far below it.")


(b) Positive control + (c) overlapping windows — 10 client-grouped splits



LEAK on purpose: + impressions last/prev 30d AUC 0.998 ± 0.003 | P@10 1.00 | P@50 1.00 | base rate 0.48


Paper features (90d windows + static)        AUC 0.701 ± 0.059 | P@10 0.87 | P@50 0.80 | base rate 0.48


Strict: days 61-90 traffic + static          AUC 0.634 ± 0.056 | P@10 0.75 | P@50 0.66 | base rate 0.48


Static only (no traffic at all)              AUC 0.536 ± 0.036 | P@10 0.57 | P@50 0.55 | base rate 0.48
Week 4 rule (reference)                      AUC 0.543         | P@10 0.39 | P@50 0.44

Positive control lit up (harness works); the clean feature set stays far below it.


In [5]:
# (d) Product flags: FlyRank's decision outputs must not exist in the data or the features
PRODUCT_FLAGS = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win",
                 "refresh_tier", "needs_content_fix", "is_zombie"]
present = [c for c in PRODUCT_FLAGS if c in raw.columns]
assert not present and not set(PRODUCT_FLAGS) & set(FEATURES), present
print(f"(d) Product flags present in data: {present or 'none'} — nothing circular to learn from")

# (e) Client memorization: same features, random rows vs whole clients held out
rand_auc = []
for seed in range(5):
    a, b = train_test_split(df, test_size=0.20, random_state=RANDOM_SEED + seed)
    s = make_model(NUMERIC_FEATURES, CATEGORICAL_FEATURES).fit(a[FEATURES], a["is_declining_label"]).predict_proba(b[FEATURES])[:, 1]
    rand_auc.append(roc_auc_score(b["is_declining_label"], s))
print(f"(e) Random-row split AUC {np.mean(rand_auc):.3f} vs client-grouped {paper['AUC_mean']:.3f} "
      f"-> {np.mean(rand_auc) - paper['AUC_mean']:+.3f} from seeing other pages of the same client")

# Privacy scan: text columns must hold no URLs, emails, domains; IDs must look like pseudonyms
pattern = re.compile(r"https?://|www\.|@|\.(?:com|net|org|io|co)\b", re.I)
text_cols = raw.select_dtypes("object").columns
hits = {c: int(raw[c].astype(str).str.contains(pattern).sum()) for c in text_cols}
assert sum(hits.values()) == 0, {c: n for c, n in hits.items() if n}
id_ok = (raw["client_id"].astype(str).str.fullmatch(r"client_[0-9a-f]{10}").all()
         and raw["content_id"].astype(str).str.fullmatch(r"content_[0-9a-f]{12}").all())
print(f"\nPrivacy scan over {len(text_cols)} text columns: 0 URL/email/domain matches")
assert id_ok, "IDs do not match the pseudonym format"
print("client_id / content_id match the pseudonym format client_<hex> / content_<hex> (grouping only; never printed here)")

summary = {
    "dataset": "data/raw/content_refresh_anonymized.csv (starter slice)", "n_client_grouped_splits": N_SPLITS,
    "single_column_suspects": suspects["column"].tolist(),
    "feature_sets": {r["feature_set"]: {k: float(v) for k, v in r.items() if k != "feature_set"} for r in results},
    "week4_rule": {k: float(v) for k, v in rr.items()},
    "random_row_split_auc": float(np.mean(rand_auc)),
    "product_flags_present": present, "privacy_pattern_hits": int(sum(hits.values())),
}
out = find("work/outputs") / "leakage_check_summary.json"
out.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\nSaved receipt: {out}")


(d) Product flags present in data: none — nothing circular to learn from


(e) Random-row split AUC 0.779 vs client-grouped 0.701 -> +0.078 from seeing other pages of the same client



Privacy scan over 14 text columns: 0 URL/email/domain matches
client_id / content_id match the pseudonym format client_<hex> / content_<hex> (grouping only; never printed here)

Saved receipt: ..\..\work\outputs\leakage_check_summary.json


**What the hunt found.**
- **Label-derived columns are real and dangerous.** `trend_pct` alone ranks the label at AUC 0.955. The 30-day window columns are only mildly informative one at a time (about 0.62), but *together* they are the label: the label is literally their ratio. None of them is in the feature set.
- **The harness works.** Adding the two impression windows together on purpose pushes AUC to 0.998, so these tests would catch leakage.
- **Overlapping windows matter: the most important finding.** The paper's 90-day features score AUC 0.70 / P@10 0.87 / P@50 0.80. Features built only from days 61–90 (strictly before the label) score 0.63 / 0.75 / 0.66. The gap is the share of the paper's performance that comes from features partly describing the label window. The strict model still beats the Week 4 rule (0.54 / 0.39 / 0.44) by a wide margin, so the conclusion "a model orders the queue better than the rule" holds. The size of the gain is smaller than the paper's headline suggests.
- **No product flags** exist in the data, so nothing circular is learned.
- **Client memorization** is real: a random row split scores higher than holding out whole clients, which is why every number above uses client-grouped splits.
- **Privacy:** no URLs, emails or domains in any text column, and IDs are pseudonyms used only for grouping.

**Action:** the capstone paper will add the strict-feature result as a sensitivity check (separate commit).


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
EXCLUDED = {
    "trend_direction": "defines the label",
    "trend_pct": "the label's own percentage change",
    "is_declining_label": "the target itself",
    "impressions_last_30d": "label window ('last 30')",
    "impressions_prev_30d": "label window ('previous 30')",
    "clicks_last_30d": "same windows as the label; near-duplicates its signal",
    "clicks_prev_30d": "same windows as the label",
    "sessions_last_30d": "same windows as the label",
    "sessions_prev_30d": "same windows as the label",
    "client_id": "pseudonym — grouping the split only; as a feature it memorizes clients",
    "content_id": "pseudonym — row identity only",
    "provider_used": "tooling metadata, not search evidence",
    "model_used": "tooling metadata, not search evidence",
}
excluded = pd.DataFrame([{"field": k, "in_data": k in df.columns, "why": v} for k, v in EXCLUDED.items()])
print(excluded.to_string(index=False))

assert not set(EXCLUDED) & set(FEATURES), set(EXCLUDED) & set(FEATURES)
used_raw = {f.replace("log_", "") for f in FEATURES}
assert not {c for c in raw.columns if c.endswith(("_last_30d", "_prev_30d"))} & used_raw
print(f"\nAssert passed: none of the {len(EXCLUDED)} excluded fields reaches the {len(FEATURES)}-feature vector.")
print("Kept but flagged (Section 2): 12 features overlap the label window; their effect is measured in Section 3c.")


               field  in_data                                                                    why
     trend_direction     True                                                      defines the label
           trend_pct     True                                      the label's own percentage change
  is_declining_label     True                                                      the target itself
impressions_last_30d     True                                               label window ('last 30')
impressions_prev_30d     True                                           label window ('previous 30')
     clicks_last_30d     True                  same windows as the label; near-duplicates its signal
     clicks_prev_30d     True                                              same windows as the label
   sessions_last_30d     True                                              same windows as the label
   sessions_prev_30d     True                                              same windows as 

**Also excluded, by design rather than by column:** FlyRank product flags (`health_score`, `priority_score`, `action_type`, …). They are not in the data, and rebuilding one as a feature would mean learning an old decision rather than the data.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.